# E-Commerce Customer Sentiment & Root Cause Analysis Pipeline
**Author:** Lidya Malak Laroum   
**Role:** Data Analyst  
**Date:** February 2026  
## Executive Summary
**The Business Problem:** The company is experiencing product returns and negative reviews, but the manual reading of 23,000+ reviews is unscalable. 
**The Solution:** This notebook engineers an automated Natural Language Processing (NLP) pipeline. It cleans raw e-commerce data, extracts sentiment polarity (how positive/negative a review is), and identifies key product failure points using NLTK keyword extraction.
**Output:** An enriched dataset prepared for executive dashboarding in Power BI.

In [1]:
!pip install textblob tqdm

In [2]:
# 1. Environment Setup & Library Imports
import pandas as pd
from textblob import TextBlob
from tqdm import tqdm
import nltk
from collections import Counter
import warnings

# Suppress harmless warnings for clean output
warnings.filterwarnings('ignore')

# Enable progress bars for pandas operations (crucial for large datasets)
tqdm.pandas()

# Download necessary NLTK models (Safe to run multiple times)
print("Downloading NLTK dependencies...")
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
print("Environment setup complete.")

Environment setup complete.


## Phase 1: Data Ingestion & Quality Assurance
**Objective:** Load the raw data and perform immediate deduplication and schema standardization. In the real world, "garbage in equals garbage out." We must ensure column names are uniform and null values in critical text fields are handled before passing them to the NLP engine.

In [3]:
# 2. Load the Dataset
input_path = "Untitled spreadsheet - Womens Clothing E-Commerce Reviews.csv" 
print(f"Loading data from {input_path}...")
df = pd.read_csv(input_path, encoding="utf-8")

# --- DATA CLEANING PIPELINE ---

# Step A: Drop exact duplicates
initial_rows = len(df)
df = df.drop_duplicates()
print(f"Dropped {initial_rows - len(df)} duplicate rows.")

# Step B: Standardize column names (snake_case for easier coding)
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

# Step C: Handle Nulls in text columns
text_cols = [c for c in df.columns if any(x in c for x in ["review", "title", "text"])]
for col in text_cols:
    df[col] = df[col].fillna("") # Fill missing text with empty strings

# Step D: Drop rows where BOTH title and review are completely empty
if len(text_cols) > 0:
    df = df[df[text_cols].apply(lambda row: any(val.strip() != "" for val in row), axis=1)]

print(f" Data Cleaned. Total valid rows remaining: {len(df)}")
df.head(2)

Loading data from Untitled spreadsheet - Womens Clothing E-Commerce Reviews.csv...
Dropped 844 duplicate rows.
 Data Cleaned. Total valid rows remaining: 22641


,transaction_id,clothing_id,age,title,review_text,rating,is_recommended,positive_feedback_count,department_name,department_name.1,class_name,review_length
0,0.0,767.0,33.0,,Absolutely wonderful - silky and sexy and comf...,4.0,1.0,0.0,Initmates,Intimate,Intimates,53.0
1,1.0,1080.0,34.0,,Love this dress! it's sooo pretty. i happene...,5.0,1.0,4.0,General,Dresses,Dresses,303.0


## Phase 2: Feature Engineering
To give our AI models the maximum amount of context, we will combine the `Title` and `Review Text` into a single feature called `full_review`. We will also calculate `review_length` to later analyze if angry customers write longer reviews than happy customers.

In [4]:
# 3. Feature Engineering

# Dynamically locate title and text columns
title_col = next((c for c in df.columns if "title" in c), None)
text_col = next((c for c in df.columns if "review_text" in c or c.endswith("review")), None)

def build_full_review(row):
    """Concatenates title and text intelligently."""
    parts = []
    if title_col and str(row[title_col]).strip() != "":
        parts.append(str(row[title_col]).strip())
    if text_col and str(row[text_col]).strip() != "":
        parts.append(str(row[text_col]).strip())
    return " - ".join(parts)

# Apply functions
df["full_review"] = df.apply(build_full_review, axis=1)
df["review_length"] = df["full_review"].astype(str).apply(len)

print("Feature Engineering Complete. Snippet:")
df[["full_review", "review_length"]].head(3)

Feature Engineering Complete. Snippet:


,full_review,review_length
0,Absolutely wonderful - silky and sexy and comf...,53
1,Love this dress! it's sooo pretty. i happene...,303
2,Some major design flaws - I had such high hope...,526


## Phase 3: The NLP Architecture (Sentiment & Keywords)
Here we define the core logic for our text analysis:
1. **Polarity (`-1.0` to `1.0`):** Measures the emotion of the text. (-1 is angry, 1 is thrilled).
2. **Subjectivity (`0.0` to `1.0`):** Measures if the review is factual (0.0) or opinion-based (1.0).
3. **Keyword Extraction:** Uses NLTK Part-of-Speech (POS) tagging to extract the top 2 Nouns/Adjectives (e.g., "zipper", "fabric") to identify *what* the customer is talking about.

In [5]:
# 4. Define NLP Functions

def get_sentiment(text_val):
    """Calculates Polarity and Subjectivity using TextBlob."""
    if not isinstance(text_val, str) or text_val.strip() == "":
        return 0.0, 0.0
    tb = TextBlob(text_val)
    return tb.sentiment.polarity, tb.sentiment.subjectivity

def get_vibe_category(polarity):
    """Business logic to categorize continuous polarity into discrete bins."""
    if polarity > 0.1: return "Positive"
    elif polarity < -0.1: return "Negative"
    else: return "Neutral"

def extract_key_keywords(text_val, top_n=2):
    """Extracts top Nouns/Adjectives to identify product flaws/highlights."""
    if not isinstance(text_val, str) or text_val.strip() == "":
        return ""
    
    tokens = nltk.word_tokenize(text_val.lower())
    try:
        tagged = nltk.pos_tag(tokens, lang="eng")
    except TypeError:
        tagged = nltk.pos_tag(tokens)

    # Filter for Nouns (NN) and Adjectives (JJ)
    candidates = [word for word, pos in tagged if pos.startswith("NN") or pos.startswith("JJ")]
    
    if not candidates: return ""
    
    counts = Counter(candidates)
    return ", ".join([w for w, c in counts.most_common(top_n)])

print("NLP Architecture Defined.")

NLP Architecture Defined.


## Phase 4: Pipeline Execution
Applying the NLP architecture to the dataset. *Note: Using `tqdm` to provide a progress bar for the executive team to monitor processing times.*

In [6]:
# 5. Run the Engine
polarities, subjectivities, vibes, keywords_list = [], [], [], []

print("Initiating NLP Processing on all reviews...")
for _, row in tqdm(df.iterrows(), total=len(df), desc="Analyzing Sentiments"):
    text_val = row["full_review"]
    
    # Extract metrics
    pol, subj = get_sentiment(text_val)
    
    # Append to lists
    polarities.append(pol)
    subjectivities.append(subj)
    vibes.append(get_vibe_category(pol))
    keywords_list.append(extract_key_keywords(text_val, top_n=2))

# Assign to dataframe
df["sentiment_polarity"] = polarities
df["sentiment_subjectivity"] = subjectivities
df["vibe_category"] = vibes
df["key_keywords"] = keywords_list

print("\n NLP Execution Complete.")

Initiating NLP Processing on all reviews...


Analyzing Sentiments: 100%|██████████| 22641/22641 [03:08<00:00, 120.27it/s]


 NLP Execution Complete.


## Phase 5: Quality Assurance & Export
Before exporting to Power BI,  We will check if our AI-generated "Polarity" logically aligns with the actual "Star Ratings" given by customers. If 1-star reviews have negative polarity, our model is accurate.

In [7]:
# 6. Sanity Checks and Export

print("--- MODEL VALIDATION ---")
print("Vibe Category Distribution:")
print(df["vibe_category"].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

if "rating" in df.columns:
    print("\nAverage Sentiment Polarity vs Actual Star Rating:")
    validation_df = df.groupby("rating")["sentiment_polarity"].mean().sort_index()
    print(validation_df)
    print("\nInsight: If Polarity increases as Rating increases, the NLP model is mathematically sound.")

# Export for Power BI
output_path = "Enriched_Womens_Clothing_Reviews.csv"
df.to_csv(output_path, index=False)
print(f"\n Pipeline Finished. Data exported successfully to: {output_path}")

--- MODEL VALIDATION ---
Vibe Category Distribution:
vibe_category
Positive    84.8%
Neutral     13.5%
Negative     1.7%
Name: proportion, dtype: object

Average Sentiment Polarity vs Actual Star Rating:
rating
1.0    0.061698
2.0    0.118254
3.0    0.170236
4.0    0.245965
5.0    0.328405
Name: sentiment_polarity, dtype: float64

Insight: If Polarity increases as Rating increases, the NLP model is mathematically sound.

 Pipeline Finished. Data exported successfully to: Enriched_Womens_Clothing_Reviews.csv
